<a href="https://colab.research.google.com/github/liliettemorejon/msis-assignments-retail-relabel/blob/main/retail_relabel_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Price-Tag Decision: Logistic Regression and Menu Costs
**ISM6642 – Machine Learning for Business Applications | Dr. Hemang Subramanian**
Team: Priscila, Liliette, Yasori

This notebook runs end-to-end from `retail_relabel.csv` and covers all five parts of the assignment:

| Step | Assignment part | What we do |
|---|---|---|
| Step 0 | Setup | Imports, constants, helper functions, data upload |
| Step 1 | Part 1 (15 pts) | Build the gap features from the three competitor prices; base rates |
| Step 2 | Part 2 (15 pts) | Model 1 – naive model with the *signed* gap |
| Step 3 | Part 3 (25 pts) | Model 2 – economically motivated model, odds ratios, model comparison |
| Step 4 | Part 4 (15 pts) | 70/30 stratified split, refit Model 2, test confusion matrix / accuracy / AUC |
| Step 5 | Part 5 (25 pts) | ExpectedWeeklyLoss, NetBenefit, share recouping cost in one week |
| Step 6 | Discussion | Evidence for the four discussion questions |
| Appendix | Liliette's detailed guide | Parts 1–2 step by step, Discussion Q2, VIF notes, reproducibility checks |

Run the cells in order. In Colab, run the upload cell in Step 0 and select `retail_relabel.csv`.

---
# Step 0: Setup

### Install and import libraries
- **statsmodels** gives the *explanatory* output the assignment asks for: coefficients, p-values, pseudo R², AIC, BIC.
- **scikit-learn** gives the *predictive* tools: the train/test split, confusion matrix, accuracy, and AUC.

In [1]:
# statsmodels is already in Colab; -q keeps the install output quiet if it is missing
!pip install -q statsmodels

import numpy as np                              # math: log, exp, absolute value
import pandas as pd                             # read the CSV and hold the data in a table
import statsmodels.formula.api as smf           # logistic regression written as a formula
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

### Constants
Following the professor's style, every setting lives here once, in UPPERCASE, so a change only needs one edit.

In [2]:
DATA_FILE = "retail_relabel.csv"
TARGET = "Relabel"                               # 1 = tag was changed this cycle, 0 = left alone

# The three competitor price columns reported by the price-intelligence feed
COMPETITORS = ["Competitor_BigBoxDepot", "Competitor_ValueMart", "Competitor_QuickShop"]

# Model formulas (statsmodels syntax: target ~ predictor1 + predictor2 + ...)
MODEL1_FORMULA = "Relabel ~ GapPct + UnitCost + RelabelCost"
MODEL2_FORMULA = ("Relabel ~ AbsGapPct + CostChangePct + LogUnits + RelabelCost"
                  " + DaysSinceLastChange + UnitCost")

# Train/test settings required by Part 4 (seed respected = reproducibility points)
TEST_SIZE = 0.30
RANDOM_STATE = 7
THRESHOLD = 0.5

### Helper functions
Like the professor's `load` and `evaluate`, each function does one job and is reused, so we never copy-paste the same code.

In [3]:
def load(path):
    """Load the dataset and build every engineered feature Part 1 asks for."""
    df = pd.read_csv(path)
    df["CompAvg"] = df[COMPETITORS].mean(axis=1)                    # average competitor price per SKU
    df["GapPct"] = 100 * (df["StorePrice"] - df["CompAvg"]) / df["CompAvg"]   # signed gap (%)
    df["AbsGapPct"] = df["GapPct"].abs()                            # size of the gap, direction ignored
    df["LogUnits"] = np.log(df["WeeklyUnitsSold"])                  # diminishing returns to velocity
    df["MarginPct"] = 100 * (df["StorePrice"] - df["UnitCost"]) / df["StorePrice"]   # gross margin (%)
    return df

In [4]:
def fit_logit(formula, data):
    """Fit a logistic regression from a formula; disp=0 hides the optimizer log."""
    return smf.logit(formula, data=data).fit(disp=0)

In [5]:
def odds_ratio_table(model):
    """Convert log-odds coefficients into odds ratios with 95% confidence intervals."""
    ci = np.exp(model.conf_int())                  # exponentiate BOTH ends of the CI
    ci.columns = ["OR_2.5%", "OR_97.5%"]
    table = pd.concat([np.exp(model.params).rename("OddsRatio"), ci,
                       model.pvalues.rename("p_value")], axis=1)
    return table.round(4)

In [6]:
def evaluate(model, data, name):
    """Print the confusion matrix, accuracy, and AUC for one dataset."""
    y = data[TARGET]
    prob = model.predict(data)                     # predicted probability of relabel
    pred = (prob >= THRESHOLD).astype(int)         # classify with the 0.5 threshold
    print(f"\n{name}")
    print("Confusion matrix (rows = actual 0/1, columns = predicted 0/1):")
    print(confusion_matrix(y, pred))
    print(f"Accuracy: {accuracy_score(y, pred):.4f}")
    print(f"AUC:      {roc_auc_score(y, prob):.4f}")
    return prob

### Upload the data
Colab only: run this cell and choose `retail_relabel.csv`. Running locally, it skips itself as long as the file is in the same folder.

In [7]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    pass    # not running in Colab

Saving retail_relabel.csv to retail_relabel.csv


---
# Step 1: Part 1 – Feature engineering and EDA (15 pts)

### Load the data and build the features
`load()` creates the five engineered columns:
- **CompAvg** – the average of the three competitor prices.
- **GapPct** – the signed gap, 100 × (StorePrice − CompAvg) / CompAvg. It is positive when we are more expensive than competitors and negative when we are cheaper.
- **AbsGapPct** – |GapPct|, the size of the misalignment in either direction.
- **LogUnits** – ln(WeeklyUnitsSold).
- **MarginPct** – 100 × (StorePrice − UnitCost) / StorePrice.

In [8]:
df = load(DATA_FILE)
print(f"Rows, columns: {df.shape}")                  # expect (1500, 18) after feature engineering
print(f"Missing values: {df.isna().sum().sum()}")    # expect 0

print("\nEngineered features (summary):")
print(df[["CompAvg", "GapPct", "AbsGapPct", "LogUnits", "MarginPct"]].describe().round(2))

Rows, columns: (1500, 18)
Missing values: 0

Engineered features (summary):
       CompAvg   GapPct  AbsGapPct  LogUnits  MarginPct
count  1500.00  1500.00    1500.00   1500.00    1500.00
mean     19.35     3.62      11.00      3.94      32.68
std      25.38    12.73       7.36      0.50       7.74
min       1.58   -28.27       0.02      1.79      16.73
25%       5.82    -6.52       4.93      3.63      26.64
50%      10.32     3.71      10.10      3.99      33.74
75%      19.47    13.41      15.61      4.29      39.32
max     184.76    42.17      42.17      5.16      44.44


### Relabel base rates: overall, by Category, by ESL
A **base rate** is the share of SKUs whose tag was changed (the mean of a 0/1 column).

In [9]:
print(f"Overall relabel rate: {df[TARGET].mean():.4f}")

print("\nBy Category:")
print(df.groupby("Category")[TARGET].agg(["mean", "count"]).round(4))

# ESL = electronic shelf label (1) vs paper tag (0); we also show the average menu cost
print("\nBy ElectronicShelfLabel (0 = paper, 1 = ESL):")
print(df.groupby("ElectronicShelfLabel")[[TARGET, "RelabelCost"]].mean().round(4))

Overall relabel rate: 0.4000

By Category:
                mean  count
Category                   
Apparel       0.3902    287
Electronics   0.4398    191
Grocery       0.4115    486
Household     0.3784    296
PersonalCare  0.3833    240

By ElectronicShelfLabel (0 = paper, 1 = ESL):
                      Relabel  RelabelCost
ElectronicShelfLabel                      
0                      0.3388       1.4477
1                      0.5407       0.0601


**Reading the output:** ESL items are relabeled far more often (≈54% vs. ≈34%). Their average relabel cost is ≈$0.06 vs. ≈$1.45 for paper tags. That is the menu cost at work: when changing a tag is nearly free, even small mispricings get corrected.

### Diagnostic: relabel rate across bins of the signed gap
This table shows *why* the functional form matters (Part 2). Look for a U-shape.

In [10]:
# pd.cut groups the continuous gap into ranges; observed=True hides empty groups
gap_bins = pd.cut(df["GapPct"], [-40, -15, -5, 5, 15, 45])
print(df.groupby(gap_bins, observed=True)[TARGET].agg(["mean", "count"]).round(4))

              mean  count
GapPct                   
(-40, -15]  0.6429     98
(-15, -5]   0.3625    331
(-5, 5]     0.1342    380
(5, 15]     0.3592    373
(15, 45]    0.7296    318


---
# Step 2: Part 2 – A naive model (15 pts)

### Model 1: `Relabel ~ GapPct + UnitCost + RelabelCost`
A logistic coefficient is the change in the **log-odds** of relabeling for a one-unit increase in that variable, holding the others constant (lecture slide 9).

In [11]:
model1 = fit_logit(MODEL1_FORMULA, df)
print(model1.summary())

print(f"\nModel 1 pseudo R-squared: {model1.prsquared:.4f}")
print(f"Model 1 AIC:              {model1.aic:.2f}")

                           Logit Regression Results                           
Dep. Variable:                Relabel   No. Observations:                 1500
Model:                          Logit   Df Residuals:                     1496
Method:                           MLE   Df Model:                            3
Date:                Sun, 27 Sep 2026   Pseudo R-squ.:                 0.07727
Time:                        05:14:10   Log-Likelihood:                -931.51
converged:                       True   LL-Null:                       -1009.5
Covariance Type:            nonrobust   LLR p-value:                 1.328e-33
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.0589      0.098      0.603      0.547      -0.133       0.250
GapPct          0.0379      0.005      8.383      0.000       0.029       0.047
UnitCost        0.0005      0.003      0.154    

**Why the signed gap is the wrong functional form:** an item 10% *over* competitors and an item 10% *under* both call for a new tag. The first loses customers; the second leaves margin on the table. A single linear term in GapPct can only say "higher gap → more (or less) likely," so it cannot represent a U-shape.

---
# Step 3: Part 3 – The economically motivated model (25 pts)

### Model 2
Model 2 replaces the signed gap with **AbsGapPct** and adds cost pressure (CostChangePct), velocity (LogUnits), and staleness (DaysSinceLastChange).

In [12]:
model2 = fit_logit(MODEL2_FORMULA, df)
print(model2.summary())

                           Logit Regression Results                           
Dep. Variable:                Relabel   No. Observations:                 1500
Model:                          Logit   Df Residuals:                     1493
Method:                           MLE   Df Model:                            6
Date:                Sun, 27 Sep 2026   Pseudo R-squ.:                  0.2750
Time:                        05:14:11   Log-Likelihood:                -731.91
converged:                       True   LL-Null:                       -1009.5
Covariance Type:            nonrobust   LLR p-value:                1.059e-116
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -4.2772      0.567     -7.540      0.000      -5.389      -3.165
AbsGapPct               0.1778      0.011     16.245      0.000       0.156       0.199
CostChangePct   

### Odds ratios with 95% confidence intervals
An odds ratio is exp(coefficient). OR > 1 raises the odds of relabeling, OR < 1 lowers them, and a CI that contains 1 means no detectable effect.

In [13]:
print(odds_ratio_table(model2))

                     OddsRatio  OR_2.5%  OR_97.5%  p_value
Intercept               0.0139   0.0046    0.0422    0.000
AbsGapPct               1.1946   1.1692    1.2205    0.000
CostChangePct           1.1361   1.0992    1.1742    0.000
LogUnits                1.5410   1.1919    1.9923    0.001
RelabelCost             0.4236   0.3585    0.5006    0.000
DaysSinceLastChange     1.0040   1.0028    1.0053    0.000
UnitCost                1.0004   0.9931    1.0077    0.919


### CostChangePct vs. UnitCost (the null-effect test)
Compare the two p-values: the **change** in cost matters, the **level** of cost does not.

In [14]:
for var in ["CostChangePct", "UnitCost"]:
    print(f"{var:15s} OR = {np.exp(model2.params[var]):.4f}   p-value = {model2.pvalues[var]:.4f}")

CostChangePct   OR = 1.1361   p-value = 0.0000
UnitCost        OR = 1.0004   p-value = 0.9190


### Model 1 vs. Model 2
Pseudo R²: higher is better. AIC and BIC: lower is better. BIC penalizes extra predictors more heavily than AIC.

In [15]:
comparison = pd.DataFrame({
    "PseudoR2": [model1.prsquared, model2.prsquared],
    "AIC":      [model1.aic, model2.aic],
    "BIC":      [model1.bic, model2.bic]},
    index=["Model 1 (signed gap)", "Model 2 (economic)"])
print(comparison.round(3))

                      PseudoR2       AIC       BIC
Model 1 (signed gap)     0.077  1871.020  1892.273
Model 2 (economic)       0.275  1477.817  1515.009


### Multicollinearity check (VIF)
VIF (Variance Inflation Factor) measures how much a predictor overlaps with the *other* predictors. A value of 1 means no overlap, and above 5 is a warning sign.

In [16]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

MODEL2_VARS = ["AbsGapPct", "CostChangePct", "LogUnits", "RelabelCost",
               "DaysSinceLastChange", "UnitCost"]
X = sm.add_constant(df[MODEL2_VARS])     # VIF needs the intercept column in the design matrix

vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
                index=MODEL2_VARS, name="VIF")
print(vif.round(2))

# Why ESL is NOT in the model alongside RelabelCost: ESL is the reason the cost is near zero
print(f"\nCorrelation ElectronicShelfLabel vs RelabelCost: "
      f"{df['ElectronicShelfLabel'].corr(df['RelabelCost']):.2f}")

AbsGapPct              1.0
CostChangePct          1.0
LogUnits               1.0
RelabelCost            1.0
DaysSinceLastChange    1.0
UnitCost               1.0
Name: VIF, dtype: float64

Correlation ElectronicShelfLabel vs RelabelCost: -0.78


---
# Step 4: Part 4 – Predictive evaluation (15 pts)

### 70/30 stratified split and refit
- `stratify` keeps the 40% relabel rate in both sets.
- `random_state=7` makes the split identical every time it runs.
- Model 2 is refit on the **training** rows only, then scored on the **test** rows it has never seen.

In [17]:
train, test = train_test_split(df, test_size=TEST_SIZE,
                               stratify=df[TARGET], random_state=RANDOM_STATE)
print(f"Train: {len(train)} rows   Test: {len(test)} rows")

model3 = fit_logit(MODEL2_FORMULA, train)
print(f"Training pseudo R-squared: {model3.prsquared:.4f}")

Train: 1050 rows   Test: 450 rows
Training pseudo R-squared: 0.2552


### Evaluate on the training and test data

In [18]:
evaluate(model3, train, "Training data")
test = test.copy()                                   # avoid pandas' SettingWithCopy warning
test["p_hat"] = evaluate(model3, test, "Test data")  # keep test probabilities for Step 5

# Baseline: accuracy of a "model" that always predicts the majority class (no relabel)
print(f"\nBaseline accuracy (always predict 0): {1 - test[TARGET].mean():.4f}")


Training data
Confusion matrix (rows = actual 0/1, columns = predicted 0/1):
[[534  96]
 [151 269]]
Accuracy: 0.7648
AUC:      0.8210

Test data
Confusion matrix (rows = actual 0/1, columns = predicted 0/1):
[[233  37]
 [ 64 116]]
Accuracy: 0.7756
AUC:      0.8533

Baseline accuracy (always predict 0): 0.6000


---
# Step 5: Part 5 – Economic decision analysis (25 pts)

### Expected weekly loss and net benefit for each test SKU
- **ExpectedWeeklyLoss** = (AbsGapPct / 100) × StorePrice × WeeklyUnitsSold. This is the dollars at stake each week while the price stays misaligned.
- **NetBenefit** = ExpectedWeeklyLoss − RelabelCost. If it is positive, one week of corrected pricing pays for the tag change.

In [19]:
test["ExpectedWeeklyLoss"] = (test["AbsGapPct"] / 100) * test["StorePrice"] * test["WeeklyUnitsSold"]
test["NetBenefit"] = test["ExpectedWeeklyLoss"] - test["RelabelCost"]

share = (test["NetBenefit"] > 0).mean()
print(f"Share of test SKUs recouping relabel cost within one week: {share:.4f}")
print(f"({(test['NetBenefit'] > 0).sum()} of {len(test)} SKUs)")

Share of test SKUs recouping relabel cost within one week: 0.9867
(444 of 450 SKUs)


### Segments for the memo: where is the money?

In [20]:
print("By Category (median $ per week):")
print(test.groupby("Category")[["ExpectedWeeklyLoss", "NetBenefit"]].median()
          .sort_values("ExpectedWeeklyLoss", ascending=False).round(2))

# Size of the gap: which misalignment bands carry the most loss?
gap_band = pd.cut(test["AbsGapPct"], [0, 5, 15, 45], labels=["0-5%", "5-15%", "15%+"])
print("\nBy size of gap (median $ per week):")
print(test.groupby(gap_band, observed=True)[["ExpectedWeeklyLoss", "NetBenefit"]].median().round(2))

# Velocity: split SKUs into four equal-size groups by weekly units sold
velocity = pd.qcut(test["WeeklyUnitsSold"], 4, labels=["Slowest 25%", "Q2", "Q3", "Fastest 25%"])
print("\nBy sales velocity (median $ per week):")
print(test.groupby(velocity, observed=True)[["ExpectedWeeklyLoss", "NetBenefit"]].median().round(2))

By Category (median $ per week):
              ExpectedWeeklyLoss  NetBenefit
Category                                    
Electronics               284.38      284.34
Apparel                   112.86      110.52
Household                  35.21       34.88
PersonalCare               30.03       28.22
Grocery                    21.62       20.62

By size of gap (median $ per week):
           ExpectedWeeklyLoss  NetBenefit
AbsGapPct                                
0-5%                    11.50       10.41
5-15%                   54.88       53.40
15%+                   125.79      124.98

By sales velocity (median $ per week):
                 ExpectedWeeklyLoss  NetBenefit
WeeklyUnitsSold                                
Slowest 25%                   23.37       21.83
Q2                            42.69       41.65
Q3                            47.98       47.91
Fastest 25%                  103.61      103.54


### Business case for electronic shelf labels
We use Model 2 to ask: *if paper-tag items had the ESL relabel cost, how much more often would they be corrected?* This is a model-based scenario, not a causal estimate (see the memo caution).

In [21]:
print("Average relabel cost and relabel rate by label type (full data):")
print(df.groupby("ElectronicShelfLabel")[["RelabelCost", TARGET]].mean().round(4))

paper = df[df["ElectronicShelfLabel"] == 0].copy()
esl_cost = df.loc[df["ElectronicShelfLabel"] == 1, "RelabelCost"].mean()
saving = paper["RelabelCost"].mean() - esl_cost

p_now = model2.predict(paper).mean()      # predicted relabel rate with paper-tag costs
paper["RelabelCost"] = esl_cost           # scenario: same items, ESL-level cost
p_esl = model2.predict(paper).mean()

print(f"\nSaving per tag change if converted: ${saving:.2f}")
print(f"Paper-tag SKUs: {len(paper)}")
print(f"Predicted relabel rate for paper-tag items: {p_now:.4f} -> {p_esl:.4f} at ESL cost")

Average relabel cost and relabel rate by label type (full data):
                      RelabelCost  Relabel
ElectronicShelfLabel                      
0                          1.4477   0.3388
1                          0.0601   0.5407

Saving per tag change if converted: $1.39
Paper-tag SKUs: 1045
Predicted relabel rate for paper-tag items: 0.3399 -> 0.5387 at ESL cost


---
# Step 6: Discussion questions – supporting evidence

### Q1: Is the gain from Model 2 really about signed vs. absolute gap?
To isolate the functional form, we refit Model 2 with the signed gap swapped in and nothing else changed.

In [22]:
model2_signed = fit_logit(MODEL2_FORMULA.replace("AbsGapPct", "GapPct"), df)
print(f"Model 2 with SIGNED gap:   pseudo R2 = {model2_signed.prsquared:.4f}   AIC = {model2_signed.aic:.2f}")
print(f"Model 2 with ABSOLUTE gap: pseudo R2 = {model2.prsquared:.4f}   AIC = {model2.aic:.2f}")

Model 2 with SIGNED gap:   pseudo R2 = 0.1251   AIC = 1780.47
Model 2 with ABSOLUTE gap: pseudo R2 = 0.2750   AIC = 1477.82


### Q2: MarginPct was computed but never used. Would it add anything?

In [23]:
model2_margin = fit_logit(MODEL2_FORMULA + " + MarginPct", df)
print(f"MarginPct coefficient: {model2_margin.params['MarginPct']:.4f}   p-value = {model2_margin.pvalues['MarginPct']:.4f}")
print(f"AIC with MarginPct: {model2_margin.aic:.2f}   without: {model2.aic:.2f}")
print(f"Correlation of MarginPct with GapPct: {df['MarginPct'].corr(df['GapPct']):.2f}")

MarginPct coefficient: -0.0011   p-value = 0.8918
AIC with MarginPct: 1479.80   without: 1477.82
Correlation of MarginPct with GapPct: 0.90


### Q3: In-sample fit vs. out-of-sample performance

In [24]:
print(f"In-sample   Model 2 pseudo R2 (all data):   {model2.prsquared:.4f}")
print(f"In-sample   Model 3 pseudo R2 (train only): {model3.prsquared:.4f}")
print(f"Out-of-sample Model 3 test AUC:             {roc_auc_score(test[TARGET], test['p_hat']):.4f}")

In-sample   Model 2 pseudo R2 (all data):   0.2750
In-sample   Model 3 pseudo R2 (train only): 0.2552
Out-of-sample Model 3 test AUC:             0.8533


### Q4: From significance to business value
The Step 5 output is the evidence. Statistical significance tells us *which* factors matter; the NetBenefit and ESL scenario translate that into dollars and an operational decision.

---
# Appendix: Liliette's detailed guide (Parts 1, 2, Discussion Q2, Reproducibility)
This appendix is the step-by-step version of Parts 1 and 2 from `Liliette_Guide_Part1_Part2.ipynb`. Its setup cells were removed because Step 0 above already installs the libraries, defines the constants, and uploads the CSV. It rebuilds `df` from the raw file one feature at a time, so its numbers should match Steps 1–2 exactly.

**Note:** the graded deliverable for GitHub is the team notebook (Steps 0–6). This combined version is for study and rerunning.


---
# Appendix – Part 1: Feature engineering and EDA (15 pts)

### 1.1 Look at the raw data first
Before creating anything, check what you have: 1,500 rows (one per SKU), 13 columns, and no missing values.

In [25]:
raw = pd.read_csv(DATA_FILE)
print(f"Rows, columns: {raw.shape}")
print(f"Missing values: {raw.isna().sum().sum()}")
print("\nColumns:", list(raw.columns))
print(raw.head())

Rows, columns: (1500, 13)
Missing values: 0

Columns: ['SKU', 'Category', 'UnitCost', 'StorePrice', 'CostChangePct', 'Competitor_BigBoxDepot', 'Competitor_ValueMart', 'Competitor_QuickShop', 'WeeklyUnitsSold', 'DaysSinceLastChange', 'ElectronicShelfLabel', 'RelabelCost', 'Relabel']
        SKU     Category  UnitCost  StorePrice  CostChangePct  \
0  SKU10000      Grocery      1.74        3.06           6.98   
1  SKU10001  Electronics     31.67       54.91           5.86   
2  SKU10002    Household     17.18       24.37           2.27   
3  SKU10003    Household     12.08       15.76          -8.49   
4  SKU10004    Household      4.75        6.97           7.41   

   Competitor_BigBoxDepot  Competitor_ValueMart  Competitor_QuickShop  \
0                    2.46                  3.34                  2.22   
1                   42.03                 52.81                 40.01   
2                   26.55                 28.43                 26.72   
3                   15.80         

✅ **Checkpoint:** `(1500, 13)` and `0` missing values.

### 1.2 Build the five features, one at a time
The team notebook does this inside `load()`. Here each feature gets its own line and explanation, so you can describe it in the write-up.

In [26]:
df = raw.copy()     # work on a copy so the raw data stays untouched

# CompAvg: the average of the three competitor prices, per SKU.
# axis=1 means "average ACROSS the columns of each row" (one number per SKU).
df["CompAvg"] = df[COMPETITORS].mean(axis=1)

# GapPct: signed percentage gap between our price and the competitor average.
#   positive -> we are MORE expensive; negative -> we are CHEAPER
df["GapPct"] = 100 * (df["StorePrice"] - df["CompAvg"]) / df["CompAvg"]

# AbsGapPct: how far off we are, ignoring direction (the assignment's recommended form)
df["AbsGapPct"] = df["GapPct"].abs()

# LogUnits: natural log of weekly units sold. The log shrinks big numbers,
# so going 5 -> 10 units counts as much as 50 -> 100 (diminishing returns).
df["LogUnits"] = np.log(df["WeeklyUnitsSold"])

# MarginPct: gross margin as a % of the shelf price
df["MarginPct"] = 100 * (df["StorePrice"] - df["UnitCost"]) / df["StorePrice"]

print(df[["StorePrice", "CompAvg", "GapPct", "AbsGapPct", "LogUnits", "MarginPct"]].head())

   StorePrice    CompAvg     GapPct  AbsGapPct  LogUnits  MarginPct
0        3.06   2.673333  14.463840  14.463840  4.262680  43.137255
1       54.91  44.950000  22.157953  22.157953  3.526361  42.323803
2       24.37  27.233333 -10.514076  10.514076  4.510860  29.503488
3       15.76  16.620000  -5.174489   5.174489  3.988984  23.350254
4        6.97   6.646667   4.864594   4.864594  3.951244  31.850789


✅ **Checkpoint:** do one row by hand. Take the first SKU, average its three competitor prices, and check that GapPct matches the formula.

Doing one row by hand is the best way to prove to the professor that you understand the feature.

In [27]:
print("Summary of the engineered features:")
print(df[["CompAvg", "GapPct", "AbsGapPct", "LogUnits", "MarginPct"]].describe().round(2))

Summary of the engineered features:
       CompAvg   GapPct  AbsGapPct  LogUnits  MarginPct
count  1500.00  1500.00    1500.00   1500.00    1500.00
mean     19.35     3.62      11.00      3.94      32.68
std      25.38    12.73       7.36      0.50       7.74
min       1.58   -28.27       0.02      1.79      16.73
25%       5.82    -6.52       4.93      3.63      26.64
50%      10.32     3.71      10.10      3.99      33.74
75%      19.47    13.41      15.61      4.29      39.32
max     184.76    42.17      42.17      5.16      44.44


✅ **Checkpoint:** GapPct mean ≈ 3.62, AbsGapPct mean ≈ 11.00, LogUnits mean ≈ 3.94, MarginPct mean ≈ 32.68.

✍️ **Your turn:**
1. Why is the GapPct mean (3.62) so much smaller than the AbsGapPct mean (11.00)?
2. What happens to positive and negative gaps when you average them?

### 1.3 Relabel base rates
The mean of a 0/1 column is the share of 1s. So `df["Relabel"].mean()` is the relabel rate.

In [28]:
print(f"Overall relabel rate: {df[TARGET].mean():.4f}")

print("\nBy Category:")
print(df.groupby("Category")[TARGET].agg(["mean", "count"]).round(4))

print("\nBy ElectronicShelfLabel (0 = paper tag, 1 = ESL), with average relabel cost:")
print(df.groupby("ElectronicShelfLabel")[[TARGET, "RelabelCost"]].agg(["mean"]).round(4))

Overall relabel rate: 0.4000

By Category:
                mean  count
Category                   
Apparel       0.3902    287
Electronics   0.4398    191
Grocery       0.4115    486
Household     0.3784    296
PersonalCare  0.3833    240

By ElectronicShelfLabel (0 = paper tag, 1 = ESL), with average relabel cost:
                     Relabel RelabelCost
                        mean        mean
ElectronicShelfLabel                    
0                     0.3388      1.4477
1                     0.5407      0.0601


✅ **Checkpoint:** overall 0.40. Categories range from ≈0.378 (Household) to ≈0.440 (Electronics). ESL ≈0.541 vs. paper ≈0.339. Average RelabelCost ≈$0.06 (ESL) vs. ≈$1.45 (paper).

✍️ **Your turn:** interpret the ESL split in terms of **menu costs** (required by Part 1).
- What is a menu cost?
- Why would a near-zero tag cost change the owner's decision on a *small* mispricing but not on a large one?
- The category rates are fairly flat. What does that suggest about what really drives relabeling?

### 1.4 The diagnostic that sets up Part 2
Group SKUs by their signed gap and look at the relabel rate in each group.

In [29]:
gap_bins = pd.cut(df["GapPct"], [-40, -15, -5, 5, 15, 45])
print(df.groupby(gap_bins, observed=True)[TARGET].agg(["mean", "count"]).round(4))

              mean  count
GapPct                   
(-40, -15]  0.6429     98
(-15, -5]   0.3625    331
(-5, 5]     0.1342    380
(5, 15]     0.3592    373
(15, 45]    0.7296    318


✅ **Checkpoint:** ≈0.64, 0.36, 0.13, 0.36, 0.73. This is a **U-shape**.

✍️ **Your turn:** describe the shape in one sentence. Include this table in the write-up; it is your strongest evidence for Part 2.

---
# Appendix – Part 2: A naive model (15 pts)

### 2.1 Fit Model 1 with the signed gap
`smf.logit(formula, data)` fits a logistic regression. `.fit(disp=0)` hides the optimizer messages.

In [30]:
model1 = smf.logit(MODEL1_FORMULA, data=df).fit(disp=0)
print(model1.summary())

                           Logit Regression Results                           
Dep. Variable:                Relabel   No. Observations:                 1500
Model:                          Logit   Df Residuals:                     1496
Method:                           MLE   Df Model:                            3
Date:                Sun, 27 Sep 2026   Pseudo R-squ.:                 0.07727
Time:                        05:14:16   Log-Likelihood:                -931.51
converged:                       True   LL-Null:                       -1009.5
Covariance Type:            nonrobust   LLR p-value:                 1.328e-33
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.0589      0.098      0.603      0.547      -0.133       0.250
GapPct          0.0379      0.005      8.383      0.000       0.029       0.047
UnitCost        0.0005      0.003      0.154    

### 2.2 Report pseudo R² and AIC (the two numbers Part 2 asks for)
- **Pseudo R²** (McFadden) measures how much better the model is than one with no predictors. For logistic models, 0.2–0.4 is considered a good fit.
- **AIC** balances fit against the number of predictors; lower is better. It is only meaningful when comparing models fit to the same data.

In [31]:
print(f"Model 1 pseudo R-squared: {model1.prsquared:.4f}")
print(f"Model 1 AIC:              {model1.aic:.2f}")

Model 1 pseudo R-squared: 0.0773
Model 1 AIC:              1871.02


✅ **Checkpoint:** pseudo R² ≈ 0.0773, AIC ≈ 1871.02. GapPct coefficient ≈ 0.038 (p < 0.001).

✍️ **Your turn:** explain *deductively* why the signed gap is the wrong functional form. The assignment's prompt: *what does an owner do about an item that is 10% under competitors versus 10% over?*

Hints to reason through (write your own answer):
- What goes wrong for the owner in each case (lost customers vs. lost margin)?
- What does a single linear term in GapPct force the model to assume about the −10% item vs. an aligned item?
- GapPct still comes out positive and significant. Look at the bin counts above: there are more overpriced SKUs (318) than deeply underpriced ones (98). Is the effect **masked** or **diluted**?

---
# Appendix – Discussion Q2: MarginPct was computed but never used

### Test it: add MarginPct to Model 2 and compare

In [32]:
model2 = smf.logit(MODEL2_FORMULA, data=df).fit(disp=0)
model2_margin = smf.logit(MODEL2_FORMULA + " + MarginPct", data=df).fit(disp=0)

print(f"MarginPct coefficient: {model2_margin.params['MarginPct']:.4f}")
print(f"MarginPct p-value:     {model2_margin.pvalues['MarginPct']:.4f}")
print(f"Pseudo R2 with / without: {model2_margin.prsquared:.4f} / {model2.prsquared:.4f}")
print(f"AIC with / without:       {model2_margin.aic:.2f} / {model2.aic:.2f}")

MarginPct coefficient: -0.0011
MarginPct p-value:     0.8918
Pseudo R2 with / without: 0.2750 / 0.2750
AIC with / without:       1479.80 / 1477.82


In [33]:
# Why does it add nothing? Check how it relates to the variables already in the model.
print(df[["MarginPct", "GapPct", "AbsGapPct", "CostChangePct", "UnitCost"]].corr().round(2))

               MarginPct  GapPct  AbsGapPct  CostChangePct  UnitCost
MarginPct           1.00    0.90       0.29           0.03      0.00
GapPct              0.90    1.00       0.46           0.04     -0.00
AbsGapPct           0.29    0.46       1.00           0.02      0.02
CostChangePct       0.03    0.04       0.02           1.00     -0.03
UnitCost            0.00   -0.00       0.02          -0.03      1.00


✅ **Checkpoint:** p ≈ 0.89. AIC rises by about 2 (1477.82 → 1479.80). The correlation of MarginPct with GapPct is ≈ 0.90.

✍️ **Your turn:**
1. Why is a 2-point AIC *increase* exactly what you would expect from adding a useless variable?
2. Why are MarginPct and GapPct so correlated? Think about what happens to margin when we price above competitors.
3. Is margin a *level* or a *change*? How does that connect to the UnitCost result in Yasori's Part 3?
4. Where *could* margin still be useful? Consider an interaction with cost changes, or using it to prioritize SKUs in the memo.

---
# Appendix – Extra: Multicollinearity check (VIF)
The code for this check runs in Step 3 above; scroll up to its output to answer the prompts below. VIF (Variance Inflation Factor) measures how much a predictor overlaps with the *other* predictors. A value of 1 means no overlap, and above 5 is a warning sign.

✅ **Checkpoint:** every VIF ≈ 1.0, and the ESL vs. RelabelCost correlation ≈ −0.78.

✍️ **Your turn:**
1. Why are the Model 2 VIFs so clean? Hint: the data are simulated.
2. Connect this to Q2. MarginPct and GapPct correlate at 0.90. What would happen to their VIFs if both were in the same model?
3. Why does leaving ElectronicShelfLabel out of the model make sense when RelabelCost is already in it?

---
# Appendix – Reproducibility (5 pts)

### Check 1: the split is identical every run
The only randomness in the project is the train/test split. With `random_state=7`, it must select exactly the same SKUs every time.

In [34]:
from sklearn.model_selection import train_test_split

# Split twice with the same seed and confirm the test SKUs are identical
_, test_a = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=7)
_, test_b = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=7)
print("Same test SKUs both times:", test_a["SKU"].tolist() == test_b["SKU"].tolist())
print(f"Test size: {len(test_a)}   Test relabel rate: {test_a[TARGET].mean():.4f}  (stratified -> 0.40)")

Same test SKUs both times: True
Test size: 450   Test relabel rate: 0.4000  (stratified -> 0.40)


✅ **Checkpoint:** `True`, 450 rows, relabel rate 0.4000.

### Check 2: the team notebook runs end-to-end
1. Open `retail_relabel.ipynb` in a **new** Colab session.
2. Choose **Runtime → Restart session and run all** and upload the CSV when asked.
3. Confirm there are no errors and the key numbers match: Model 1 AIC ≈ 1871.02, Model 2 AIC ≈ 1477.82, test AUC ≈ 0.8533, share recouping ≈ 0.9867.

### Check 3: push to GitHub under `/msis-assignments/retail-relabel`
**Easiest route, from Colab:**
1. Open `retail_relabel.ipynb` and choose **File → Save a copy in GitHub**.
2. Pick the course repo and set the file path to `msis-assignments/retail-relabel/retail_relabel.ipynb`.

**Or on github.com:**
1. Open the repo and choose **Add file → Upload files**.
2. Type `msis-assignments/retail-relabel/` in front of the file name. GitHub creates the folders.
3. Upload the notebook and, later, the PDF from Priscila.

Also:
- Add a short `README.md` explaining how to run the notebook: open it in Colab, run all, and upload `retail_relabel.csv` when asked.
- Ask the team (or the professor) whether the CSV itself belongs in the repo.

---
# Appendix – Before you send your part to Priscila
- [ ] All ✅ checkpoints matched
- [ ] Part 1 write-up: feature definitions, base-rate table, ESL / menu-cost interpretation, U-shape table
- [ ] Part 2 write-up: pseudo R² and AIC, plus the functional-form critique (10% under vs. 10% over)
- [ ] Q2 answer on MarginPct
- [ ] Reproducibility checks done and the notebook pushed to GitHub
- [ ] Ready to explain every line above out loud